# Teletext Explorer

Explore scraped pages, synthetic data, rendered outputs, and vocabulary statistics.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from config import *
from teletext.vocab import Vocabulary
from teletext.renderer import render_page
from teletext.synthetic import generate_page
from teletext.utils import find_grid_files, load_grids, token_counts

In [ ]:
# Load or build vocabulary
if VOCAB_PATH.exists():
    vocab = Vocabulary.load(VOCAB_PATH)
    print(f"Loaded vocab: {vocab.observed_count} observed / {vocab.size} total")
else:
    vocab = Vocabulary.build_full()
    print(f"Built full vocab: {vocab.observed_count} types")

In [ ]:
# Generate and display a synthetic page
grid = generate_page(vocab)
img = render_page(grid, vocab)
print(f"Grid shape: {grid.shape}, dtype: {grid.dtype}")
print(f"Token ID range: {grid.min()} - {grid.max()}")
print(f"Image size: {img.size}")
display(img)

In [ ]:
# Load scraped page if available
raw_pngs = list(RAW_DIR.glob("*.png"))
if raw_pngs:
    img = Image.open(raw_pngs[0])
    print(f"Scraped page: {raw_pngs[0].name} ({img.size[0]}x{img.size[1]})")
    display(img)
else:
    print("No scraped pages found in", RAW_DIR)

In [ ]:
# Vocabulary statistics
grid_paths = find_grid_files([TOKENS_DIR, SYNTHETIC_DIR])
print(f"Found {len(grid_paths)} grid files")

if grid_paths:
    grids = load_grids(grid_paths[:10])
    counts = token_counts(grids)
    real_ids = [i for i in range(len(counts)) if counts[i] > 0 and i >= FIRST_REAL_TOKEN_ID]
    print(f"\nToken ID distribution (first {len(grid_paths)} grids):")
    print(f"  Non-zero real token types: {len(real_ids)}")
    
    top_n = 20
    top_ids = np.argsort(counts)[-top_n:][::-1]
    print(f"\nTop {top_n} most common token IDs:")
    for tid in top_ids:
        token = vocab.id_to_token(tid)
        if token:
            label = chr(token.char_id + 32) if token.char_id <= 95 else f"M{token.char_id - 96}"
            print(f"  ID {tid:4d} | '{label}' | fg={token.fg} bg={token.bg} | count={int(counts[tid]):,}")

In [ ]:
# Visualize color usage
if grid_paths:
    color_matrix = np.zeros((8, 8), dtype=np.float32)
    for tid, count in enumerate(counts):
        if count > 0:
            token = vocab.id_to_token(tid)
            if token:
                color_matrix[token.fg, token.bg] += count
    
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(color_matrix, cmap='viridis')
    ax.set_xlabel('Background color')
    ax.set_ylabel('Foreground color')
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.set_title(f'Color co-occurrence (log scale)')
    plt.colorbar(im, ax=ax, label='Count')
    plt.show()

In [ ]:
# Render several synthetic pages in a grid
n = 8
fig, axes = plt.subplots(2, 4, figsize=(20, 6))
for ax in axes.flatten():
    grid = generate_page(vocab)
    img = render_page(grid, vocab)
    ax.imshow(np.array(img))
    ax.axis('off')
plt.tight_layout()
plt.show()